# Improving Indonesian Emotion Detection with OpenAI o4-mini Text Normalization

In [ ]:
import pandas as pd

import torch
from torch import optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch import nn

In [ ]:
import random
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

## Data

In [ ]:
DATA_DIR = "data/normalized"
DELIMITER = ","

anger = pd.read_csv(f"{DATA_DIR}/anger.csv", delimiter=DELIMITER)
fear = pd.read_csv(f"{DATA_DIR}/fear.csv", delimiter=DELIMITER)
joy = pd.read_csv(f"{DATA_DIR}/joy.csv", delimiter=DELIMITER)
love = pd.read_csv(f"{DATA_DIR}/love.csv", delimiter=DELIMITER)
neutral = pd.read_csv(f"{DATA_DIR}/neutral.csv", delimiter=DELIMITER)
sad = pd.read_csv(f"{DATA_DIR}/sad.csv", delimiter=DELIMITER)

In [ ]:
data = pd.concat([anger, fear, joy, love, neutral, sad])

In [ ]:
data.loc[data[data['Label'] == ' Anger'].index, 'Label'] = 'Anger'

In [ ]:
data['Label'].value_counts()

In [ ]:
data

## Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size = 0.3, stratify = data['Label'], random_state = 42)

In [ ]:
train = train.reset_index(drop = True)
test = test.reset_index(drop = True)

In [ ]:
class DocumentSentimentDataset(Dataset):
    LABEL2INDEX = {'Anger': 0, 'Fear': 1, 'Joy': 2, 'Love': 3, 'Neutral': 4, 'Sad': 5}
    INDEX2LABEL = {0: 'Anger', 1: 'Fear', 2: 'Joy', 3: 'Love', 4: 'Neutral', 5: 'Sad'}
    NUM_LABELS = 6

    def load_dataset(self, path):
        df = path.copy()
        df.columns = ['Tweet', 'Label']
        df['Label'] = df['Label'].apply(lambda lab: self.LABEL2INDEX[lab])
        return df

    def __init__(self, dataset_path, tokenizer, no_special_token=False, *args, **kwargs):
        self.data = self.load_dataset(dataset_path)
        self.tokenizer = tokenizer
        self.no_special_token = no_special_token

    def __getitem__(self, index):
        data = self.data.loc[index, :]
        text, sentiment = data['Tweet'], data['Label']
        subwords = self.tokenizer.encode(text, add_special_tokens=not self.no_special_token)
        return np.array(subwords), np.array(sentiment), data['Label']

    def __len__(self):
        return len(self.data)


class DocumentSentimentDataLoader(DataLoader):
    def __init__(self, max_seq_len=512, *args, **kwargs):
        super(DocumentSentimentDataLoader, self).__init__(*args, **kwargs)
        self.collate_fn = self._collate_fn
        self.max_seq_len = max_seq_len

    def _collate_fn(self, batch):
        batch_size = len(batch)
        max_seq_len = max(map(lambda x: len(x[0]), batch))
        max_seq_len = min(self.max_seq_len, max_seq_len)

        subword_batch = np.zeros((batch_size, max_seq_len), dtype=np.int64)
        mask_batch = np.zeros((batch_size, max_seq_len), dtype=np.float32)
        sentiment_batch = np.zeros((batch_size, 1), dtype=np.int64)

        seq_list = []
        for i, (subwords, sentiment, raw_seq) in enumerate(batch):
            subwords = subwords[:max_seq_len]
            subword_batch[i, :len(subwords)] = subwords
            mask_batch[i, :len(subwords)] = 1
            sentiment_batch[i, 0] = sentiment
            seq_list.append(raw_seq)

        return subword_batch, mask_batch, sentiment_batch, seq_list

In [ ]:
import random
import numpy as np

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def count_param(module, trainable=False):
    if trainable:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    else:        
        return sum(p.numel() for p in module.parameters())

def metrics_to_string(metric_dict):
    string_list = []
    for key, value in metric_dict.items():
        string_list.append('{}:{:.2f}'.format(key, value))
    return ' '.join(string_list)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(26092020)

In [ ]:
from sklearn.metrics import f1_score

def document_sentiment_metrics_fn(list_hyp, list_label):
    metrics = {}
    metrics["F1"] = f1_score(list_label, list_hyp, average='macro')
    return metrics

In [ ]:
def forward_sequence_classification(model, batch_data, i2w, is_test=False, device='cpu', **kwargs):
    # Unpack batch data
    if len(batch_data) == 3:
        (subword_batch, mask_batch, label_batch) = batch_data
        token_type_batch = None
    elif len(batch_data) == 4:
        (subword_batch, mask_batch, token_type_batch, label_batch) = batch_data

    # Prepare input & label
    subword_batch = torch.LongTensor(subword_batch)
    mask_batch = torch.FloatTensor(mask_batch)
    token_type_batch = torch.LongTensor(token_type_batch) if token_type_batch is not None else None
    label_batch = torch.LongTensor(label_batch)

    if device == "cuda":
        subword_batch = subword_batch.cuda()
        mask_batch = mask_batch.cuda()
        token_type_batch = token_type_batch.cuda() if token_type_batch is not None else None
        label_batch = label_batch.cuda()

    # Forward model
    #outputs = model(subword_batch, attention_mask=mask_batch, token_type_ids=token_type_batch, labels=label_batch)
    outputs = model(subword_batch, attention_mask=mask_batch, labels=label_batch)
    loss, logits = outputs[:2]

    # generate prediction & label list
    list_hyp = []
    list_label = []
    hyp = torch.topk(logits, 1)[1]
    for j in range(len(hyp)):
        list_hyp.append(i2w[hyp[j].item()])
        list_label.append(i2w[label_batch[j][0].item()])

    return loss, logits, list_hyp, list_label

In [ ]:
from tqdm import tqdm

def predict(text, model):
  sentiments = []
  scores = []

  with torch.no_grad():
    for index in tqdm(text.index):
      subwords = tokenizer.encode(text[index], padding = True, truncation = True, max_length = 1024, add_special_tokens = True)
      subwords = torch.LongTensor(subwords).view(1, -1).to(model.device)

      logits = model(subwords)[0]
      label = torch.topk(logits, k=1, dim=-1)[1].squeeze().item()
      sentiments.append(i2w[label])
      scores.append(F.softmax(logits, dim=-1).squeeze()[label])
        
    return sentiments, scores

#sentimen, skor = predict(test['Tweet'])

## L1 Logistic Regression with TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer()

x_train_tfidf = tfidf.fit_transform(train['Tweet'])
x_test_tfidf = tfidf.transform(test['Tweet'])

logistic_regression = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    max_iter=2000,
    random_state=42
)

logistic_regression.fit(x_train_tfidf, train['Label'])
logistic_prediction = logistic_regression.predict(x_test_tfidf)

In [ ]:
print(classification_report(
    test['Label'],
    logistic_prediction,
    digits=4
))

## IndoBERT

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p2")
model = BertForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p2",
    num_labels=6
)

In [ ]:
train_dataset = DocumentSentimentDataset(train, tokenizer)
train_data_loader = DocumentSentimentDataLoader(
    dataset=train_dataset,
    max_seq_len=512,
    batch_size=8,
    shuffle=True
)

w2i = DocumentSentimentDataset.LABEL2INDEX
i2w = DocumentSentimentDataset.INDEX2LABEL

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=3e-6)
model.cuda()

best_f1 = 0
n_epochs = 40

for epoch in range(n_epochs):
    model.train()
    torch.set_grad_enabled(True)

    total_train_loss = 0
    list_hyp, list_label = [], []

    train_pbar = tqdm(train_data_loader, leave=True, total=len(train_data_loader))

    for i, batch_data in enumerate(train_pbar):
        loss, logits, batch_hyp, batch_label = forward_sequence_classification(
            model,
            batch_data[:-1],
            i2w=i2w,
            device='cuda'
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tr_loss = loss.item()
        total_train_loss = total_train_loss + tr_loss

        list_hyp += batch_hyp
        list_label += batch_label

        train_pbar.set_description(
            "(Epoch {}) TRAIN LOSS:{:.4f} LR:{:.8f}".format(
                (epoch + 1),
                total_train_loss / (i + 1),
                get_lr(optimizer)
            )
        )

    metrics = document_sentiment_metrics_fn(list_hyp, list_label)

    print(
        "(Epoch {}) TRAIN LOSS:{:.4f} {} LR:{:.8f}".format(
            (epoch + 1),
            total_train_loss / (i + 1),
            metrics_to_string(metrics),
            get_lr(optimizer)
        )
    )

    sentimen, skor = predict(test['Tweet'], model)

    print(
        "(Epoch {}) TEST F1 SCORE: {}\n".format(
            epoch + 1,
            f1_score(test['Label'], sentimen, average='macro')
        )
    )

torch.save(model.state_dict(), "results/indobert_model.pth")

In [ ]:
sentimen, skor = predict(test['Tweet'], model)
print(f1_score(sentimen, test['Label'], average='macro'))

0.8583259012898008


In [ ]:
print(classification_report(sentimen, test['Label'], digits=4))

              precision    recall  f1-score   support

       Anger     0.8609    0.8973    0.8787       331
        Fear     0.8787    0.8102    0.8430       295
         Joy     0.8665    0.8317    0.8487       398
        Love     0.8855    0.9054    0.8953       222
     Neutral     0.8417    0.8737    0.8574       578
         Sad     0.8295    0.8241    0.8268       307

    accuracy                         0.8569      2131
   macro avg     0.8604    0.8571    0.8583      2131
weighted avg     0.8572    0.8569    0.8566      2131



In [ ]:
indobert_model = model
indobert_tokenizer = tokenizer

## IndoBERTweet

In [ ]:
tokenizer = BertTokenizer.from_pretrained("indolem/indobertweet-base-uncased")
model = BertForSequenceClassification.from_pretrained(
    "indolem/indobertweet-base-uncased",
    num_labels=6
)

In [ ]:
train_dataset = DocumentSentimentDataset(train, tokenizer)
train_data_loader = DocumentSentimentDataLoader(
    dataset=train_dataset,
    max_seq_len=512,
    batch_size=8,
    shuffle=True
)

w2i = DocumentSentimentDataset.LABEL2INDEX
i2w = DocumentSentimentDataset.INDEX2LABEL

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=3e-6)
model.cuda()

best_f1 = 0
n_epochs = 40

for epoch in range(n_epochs):
    model.train()
    torch.set_grad_enabled(True)

    total_train_loss = 0
    list_hyp, list_label = [], []

    train_pbar = tqdm(train_data_loader, leave=True, total=len(train_data_loader))

    for i, batch_data in enumerate(train_pbar):
        loss, logits, batch_hyp, batch_label = forward_sequence_classification(
            model,
            batch_data[:-1],
            i2w=i2w,
            device='cuda'
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tr_loss = loss.item()
        total_train_loss = total_train_loss + tr_loss

        list_hyp += batch_hyp
        list_label += batch_label

        train_pbar.set_description(
            "(Epoch {}) TRAIN LOSS:{:.4f} LR:{:.8f}".format(
                (epoch + 1),
                total_train_loss / (i + 1),
                get_lr(optimizer)
            )
        )

    metrics = document_sentiment_metrics_fn(list_hyp, list_label)

    print(
        "(Epoch {}) TRAIN LOSS:{:.4f} {} LR:{:.8f}".format(
            (epoch + 1),
            total_train_loss / (i + 1),
            metrics_to_string(metrics),
            get_lr(optimizer)
        )
    )

    sentimen, skor = predict(test['Tweet'], model)

    print(
        "(Epoch {}) TEST F1 SCORE: {}\n".format(
            epoch + 1,
            f1_score(test['Label'], sentimen, average='macro')
        )
    )

torch.save(model.state_dict(), "results/indobertweet_model.pth")

In [ ]:
sentimen, skor = predict(test['Tweet'], model)
print(f1_score(sentimen, test['Label'], average='macro'))

0.8562337102434437


In [ ]:
print(classification_report(sentimen, test['Label'], digits=4))

              precision    recall  f1-score   support

       Anger     0.8464    0.8848    0.8652       330
        Fear     0.8897    0.8374    0.8627       289
         Joy     0.8089    0.8729    0.8397       354
        Love     0.9031    0.8761    0.8894       234
     Neutral     0.8433    0.8562    0.8497       591
         Sad     0.8689    0.7958    0.8307       333

    accuracy                         0.8536      2131
   macro avg     0.8600    0.8539    0.8562      2131
weighted avg     0.8549    0.8536    0.8536      2131



In [ ]:
indobertweet_model = model
indobertweet_tokenizer = tokenizer

## Embedding Analysis

In [ ]:
indobert_model.eval()
indobert_model.cuda()

embeddings_indobert = []

with torch.no_grad():
    for index in tqdm(test.index):
        subwords = indobert_tokenizer.encode(
            test['Tweet'][index],
            padding=True,
            truncation=True,
            max_length=512,
            add_special_tokens=True
        )

        subwords = torch.LongTensor(subwords).view(1, -1).cuda()
        pooler_output = indobert_model.bert(subwords).pooler_output
        embeddings_indobert.append(
            pooler_output.detach().cpu().numpy().squeeze()
        )

In [ ]:
indobertweet_model.eval()
indobertweet_model.cuda()

embeddings_indobertweet = []

with torch.no_grad():
    for index in tqdm(test.index):
        subwords = indobertweet_tokenizer.encode(
            test['Tweet'][index],
            padding=True,
            truncation=True,
            max_length=512,
            add_special_tokens=True
        )

        subwords = torch.LongTensor(subwords).view(1, -1).cuda()
        pooler_output = indobertweet_model.bert(subwords).pooler_output
        embeddings_indobertweet.append(
            pooler_output.detach().cpu().numpy().squeeze()
        )

### PCA: IndoBERT and IndoBERTweet

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pca = PCA(n_components=2)
pca_indobert = pca.fit_transform(embeddings_indobert)

print(
    "IndoBERT silhouette score:",
    silhouette_score(pca_indobert, test['Label'])
)

In [ ]:
pca = PCA(n_components=2)
pca_indobertweet = pca.fit_transform(embeddings_indobertweet)

print(
    "IndoBERTweet silhouette score:",
    silhouette_score(pca_indobertweet, test['Label'])
)

### SVD: TF-IDF

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=42)
svd_tfidf = svd.fit_transform(x_test_tfidf)

print(
    "TF-IDF silhouette score:",
    silhouette_score(svd_tfidf, test['Label'])
)

In [ ]:
import matplotlib.pyplot as plt

labels = test['Label'].reset_index(drop=True)

plt.figure(figsize=(7, 5))
for label in labels.unique():
    index = labels[labels == label].index
    plt.scatter(
        pca_indobert[index, 0],
        pca_indobert[index, 1],
        s=10,
        label=label
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("IndoBERT Embeddings")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for label in labels.unique():
    index = labels[labels == label].index
    plt.scatter(
        pca_indobertweet[index, 0],
        pca_indobertweet[index, 1],
        s=10,
        label=label
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("IndoBERTweet Embeddings")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for label in labels.unique():
    index = labels[labels == label].index
    plt.scatter(
        svd_tfidf[index, 0],
        svd_tfidf[index, 1],
        s=10,
        label=label
    )

plt.xlabel("SVD 1")
plt.ylabel("SVD 2")
plt.title("TF-IDF Representation")
plt.legend()
plt.show()